# Demo notebook (pure Python)

Here we **compare differentially private mechanisms** (random projection and baselines) on a regression dataset: we run the benchmark harness, build reports, and inspect results.

This notebook uses the **library API** directly (no subprocess): it adds `src/` to `sys.path`, loads a YAML config, calls `run_demo(cfg)`, and writes run outputs under the repo's `data/` directory. You can also run the same flow from the terminal with `scripts/run_demo.py` and `scripts/build_report.py`.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
os.chdir(REPO_ROOT)

SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = REPO_ROOT / 'data'
RUNS_DIR = DATA_DIR / 'outputs' / 'runs'
REPORT_DIR = REPO_ROOT / 'reports'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [2]:
# Small dataset switch section
DATASET_NAME = 'synthetic_redundant_regression'  # change to 'bike_sharing' if needed

CONFIG_PATHS = {
    'autompg': REPO_ROOT / 'configs' / 'demo' / 'autompg.yaml',
    'bike_sharing': REPO_ROOT / 'configs' / 'demo' / 'bike_sharing.yaml',
    'synthetic_redundant_regression': REPO_ROOT / 'configs' / 'demo' / 'synthetic_redundant_regression.yaml',
}

if DATASET_NAME not in CONFIG_PATHS:
    raise ValueError(f'Unsupported dataset: {DATASET_NAME}. Choose one of {list(CONFIG_PATHS)}')

CONFIG_PATH = CONFIG_PATHS[DATASET_NAME]
print('Selected dataset:', DATASET_NAME)
print('Config path:', CONFIG_PATH)


Selected dataset: synthetic_redundant_regression
Config path: /home/wei402/Desktop/rp-benchmark/configs/demo/synthetic_redundant_regression.yaml


In [3]:
from rpbench.config import load_config
from rpbench.runners.run_demo import run_demo
from rpbench.utils.io import save_jsonl

cfg = load_config(CONFIG_PATH)
cfg.output_root = str(RUNS_DIR)
base_seed, trial_seeds = cfg.resolve_trial_seeds()
print('Seed batch:', {
    'mode': cfg.seed_batch.mode,
    'base_seed': base_seed,
    'count': len(trial_seeds),
    'expanded_seeds': trial_seeds,
})

records = run_demo(cfg)

out_path = Path(cfg.output_root) / 'demo_results.jsonl'
save_jsonl(records, out_path)

print(f'Saved {len(records)} records to {out_path}')


Seed batch: {'mode': 'fixed', 'base_seed': 0, 'count': 8, 'expanded_seeds': [2968811710, 3964924996, 3141116543, 2613022947, 1874364848, 161328693, 2617721224, 1369798745]}
  [1/64] Mech_RP eps=0.1 trial=0 seed=2968811710
  [2/64] Mech_RP eps=0.1 trial=1 seed=3964924996
  [3/64] Mech_RP eps=0.1 trial=2 seed=3141116543
  [4/64] Mech_RP eps=0.1 trial=3 seed=2613022947
  [5/64] Mech_RP eps=0.1 trial=4 seed=1874364848
  [6/64] Mech_RP eps=0.1 trial=5 seed=161328693
  [7/64] Mech_RP eps=0.1 trial=6 seed=2617721224
  [8/64] Mech_RP eps=0.1 trial=7 seed=1369798745
  [9/64] Mech_RP eps=0.25 trial=0 seed=2968811710
  [10/64] Mech_RP eps=0.25 trial=1 seed=3964924996
  [11/64] Mech_RP eps=0.25 trial=2 seed=3141116543
  [12/64] Mech_RP eps=0.25 trial=3 seed=2613022947
  [13/64] Mech_RP eps=0.25 trial=4 seed=1874364848
  [14/64] Mech_RP eps=0.25 trial=5 seed=161328693
  [15/64] Mech_RP eps=0.25 trial=6 seed=2617721224
  [16/64] Mech_RP eps=0.25 trial=7 seed=1369798745
  [17/64] Mech_RP eps=0.5 tria

In [4]:
from rpbench.reporting.tables import build_summary_table
from rpbench.reporting.figures import ols_plot_eps_vs_mse, plot_eps_vs_covariance_error
from rpbench.reporting.summary import write_summary
from rpbench.utils.io import load_jsonl

input_path = RUNS_DIR / 'demo_results.jsonl'
rows = load_jsonl(input_path)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = REPORT_DIR / 'summary_table.csv'
ols_fig_path = REPORT_DIR / 'ols_plot_eps_vs_mse.png'
covariance_fig_path = REPORT_DIR / 'covariance_plot_eps_vs_error.png'
md_path = REPORT_DIR / 'demo_summary.md'

build_summary_table(rows, csv_path)
ols_plot_eps_vs_mse(rows, ols_fig_path)
plot_eps_vs_covariance_error(rows, covariance_fig_path)
write_summary(rows, md_path, ols_fig_path.name, covariance_fig_path.name, csv_path.name)

print('Wrote:')
print(' -', csv_path)
print(' -', ols_fig_path)
print(' -', covariance_fig_path)
print(' -', md_path)


Wrote:
 - /home/wei402/Desktop/rp-benchmark/reports/summary_table.csv
 - /home/wei402/Desktop/rp-benchmark/reports/ols_plot_eps_vs_mse.png
 - /home/wei402/Desktop/rp-benchmark/reports/covariance_plot_eps_vs_error.png
 - /home/wei402/Desktop/rp-benchmark/reports/demo_summary.md


In [5]:
import pandas as pd

summary_csv = REPORT_DIR / 'summary_table.csv'
summary_md = REPORT_DIR / 'demo_summary.md'
ols_fig_png = REPORT_DIR / 'ols_plot_eps_vs_mse.png'
covariance_fig_png = REPORT_DIR / 'covariance_plot_eps_vs_error.png'

display(pd.read_csv(summary_csv))

print('\n--- demo_summary.md ---\n')
print(summary_md.read_text())

print('\nOLS figure path:', ols_fig_png)
print('Covariance figure path:', covariance_fig_png)


,mechanism,epsilon,test_mse_mean,test_mse_std,rel_fro_mean,rel_fro_std,runtime_mean,n_seeds
0,Mech_RP,0.10,0.902085,0.667373,1.111040,0.209009,0.003932,8
1,Mech_RP,0.25,0.582345,0.417989,0.900762,0.167450,0.003928,8
2,Mech_RP,0.50,0.396012,0.283633,0.823628,0.148308,0.003925,8
3,Mech_RP,1.00,0.287619,0.201415,0.782212,0.134892,0.003931,8
4,Mech_RP_Pois,0.10,0.668533,0.391545,0.760166,0.025802,0.001487,8
5,Mech_RP_Pois,0.25,0.462658,0.236981,0.742330,0.024404,0.001469,8
6,Mech_RP_Pois,0.50,0.354712,0.197120,0.737555,0.024185,0.001465,8
7,Mech_RP_Pois,1.00,0.293081,0.182914,0.735517,0.024329,0.001455,8



--- demo_summary.md ---

# WF-001 Demo Run Summary

**Dataset:** synthetic_redundant_regression  
**Mechanisms:** Mech_RP, Mech_RP_Pois  
**Task:** OLSFromRelease  
**Epsilon grid:** [0.1, 0.25, 0.5, 1.0]  
**Delta:** 1.00e-06  
**Seed batch:** mode=fixed, base_seed=0, count=8  
**Expanded seeds:** [161328693, 1369798745, 1874364848, 2613022947, 2617721224, 2968811710, 3141116543, 3964924996]  
**Trial indices:** [0, 1, 2, 3, 4, 5, 6, 7]  
**Total records:** 65  

**Non-private baseline test MSE:** 0.051808

## Results

See [summary_table.csv](summary_table.csv) for the aggregate table.

### OLS Downstream Task

![OLS Downstream Utility Plot](ols_plot_eps_vs_mse.png)

### Covariance Release Quality

![Covariance Release Quality Plot](covariance_plot_eps_vs_error.png)

## Outputs

- Row-level results: `data/outputs/runs/demo_results.jsonl`
- Summary table: `summary_table.csv`
- OLS figure: `ols_plot_eps_vs_mse.png`
- Covariance figure: `covariance_plot_eps_vs_error.png`


OLS figure pa